# 07 Correlation Analysis

This notebook closes the Silver EDA layer by building a compact driver-session feature matrix and checking the strongest relationships before Gold feature engineering.

The goal is not to train a production model here. The goal is to identify useful signals, redundant variables, and possible leakage risks while keeping telemetry usage memory-safe.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px
import pyarrow.parquet as pq

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "07_correlation_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

print("=" * 72)
print(f"SILVER EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER EDA - 07_correlation_analysis
Start time: 2026-06-02 00:46:45.313238
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


## 1. Compact Driver-Session Feature Matrix

The matrix is built at one row per `session_key + driver_number`. This grain is the correct bridge between cleaned operational data and Gold modeling because race outcomes, grid position, lap summaries, stint behavior, weather context, and sampled telemetry can all be aligned without reading full telemetry files.

In [2]:
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
sessions = pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet")
meetings = pd.read_parquet(CLEANED_DATA_PATH / "meetings.parquet")
laps = pd.read_parquet(CLEANED_DATA_PATH / "laps.parquet")
starting_grid = pd.read_parquet(CLEANED_DATA_PATH / "starting_grid.parquet")
weather = pd.read_parquet(CLEANED_DATA_PATH / "weather.parquet")
stints = pd.read_parquet(CLEANED_DATA_PATH / "stints.parquet")
overtakes = pd.read_parquet(CLEANED_DATA_PATH / "overtakes.parquet")

sessions["event_type"] = np.where(
    sessions["session_name"].astype(str).str.lower().eq("sprint"),
    "SPRINT_RACE",
    "GRAND_PRIX_RACE",
)

base = session_result.copy()
base["finish_pos"] = pd.to_numeric(base["position"], errors="coerce")
base["target_win"] = (base["finish_pos"] == 1).astype(int)
base["target_podium"] = (base["finish_pos"] <= 3).astype(int)
base["target_top10"] = (base["finish_pos"] <= 10).astype(int)
base["is_dnf_like"] = base[["dnf", "dns", "dsq"]].any(axis=1) | base["finish_pos"].isna()

driver_dim = drivers[["session_key", "driver_number", "full_name", "team_name"]].drop_duplicates(["session_key", "driver_number"])
base = base.merge(driver_dim, on=["session_key", "driver_number"], how="left")
base = base.merge(sessions[["session_key", "year", "event_type", "meeting_key", "circuit_short_name", "country_name"]], on="session_key", how="left", suffixes=("", "_session"))
base["driver_id"] = base["full_name"].fillna("Driver " + base["driver_number"].astype(str))

lap_work = laps.copy()
lap_work["lap_duration"] = pd.to_numeric(lap_work["lap_duration"], errors="coerce")
lap_work = lap_work[lap_work["lap_duration"].between(50, 900)]
lap_features = lap_work.groupby(["session_key", "driver_number"], as_index=False).agg(
    avg_lap=("lap_duration", "mean"),
    median_lap=("lap_duration", "median"),
    best_lap=("lap_duration", "min"),
    std_lap=("lap_duration", "std"),
    laps_completed=("lap_number", "nunique"),
    outlier_laps=("is_outlier_lap", "sum"),
    pit_out_laps=("is_pit_out_lap", "sum"),
    avg_i1_speed=("i1_speed", "mean"),
    avg_i2_speed=("i2_speed", "mean"),
    avg_st_speed=("st_speed", "mean"),
)

grid = starting_grid[["session_key", "driver_number", "position"]].copy()
grid["grid_pos"] = pd.to_numeric(grid["position"], errors="coerce")
grid = grid.drop(columns=["position"])

weather_session = weather.groupby("session_key", as_index=False).agg(
    air_temperature=("air_temperature", "mean"),
    track_temperature=("track_temperature", "mean"),
    humidity=("humidity", "mean"),
    pressure=("pressure", "mean"),
    wind_speed=("wind_speed", "mean"),
    rainfall=("rainfall", "max"),
)

stint_features = stints.groupby(["session_key", "driver_number"], as_index=False).agg(
    stint_count=("stint_number", "nunique"),
    max_tyre_age_start=("tyre_age_at_start", "max"),
    avg_tyre_age_start=("tyre_age_at_start", "mean"),
)

overtake_for = overtakes.groupby(["session_key", "overtaking_driver_number"]).size().reset_index(name="overtakes_made")
overtake_for = overtake_for.rename(columns={"overtaking_driver_number": "driver_number"})
overtake_against = overtakes.groupby(["session_key", "overtaken_driver_number"]).size().reset_index(name="overtaken_count")
overtake_against = overtake_against.rename(columns={"overtaken_driver_number": "driver_number"})

master = (
    base
    .merge(lap_features, on=["session_key", "driver_number"], how="left")
    .merge(grid, on=["session_key", "driver_number"], how="left")
    .merge(weather_session, on="session_key", how="left")
    .merge(stint_features, on=["session_key", "driver_number"], how="left")
    .merge(overtake_for, on=["session_key", "driver_number"], how="left")
    .merge(overtake_against, on=["session_key", "driver_number"], how="left")
)
master[["overtakes_made", "overtaken_count"]] = master[["overtakes_made", "overtaken_count"]].fillna(0)
master["positions_gained"] = master["grid_pos"] - master["finish_pos"]
master["points_per_lap"] = master["points"] / master["laps_completed"].replace(0, np.nan)

telemetry_path = ROOT / "eda" / "silver" / "outputs" / "tables" / "06_telemetry_analysis" / "telemetry_result_join.csv"
if telemetry_path.exists():
    telemetry_sample = pd.read_csv(telemetry_path)
    telemetry_cols = [
        "session_key", "driver_number", "avg_speed", "max_speed", "std_speed",
        "avg_rpm", "avg_throttle", "avg_brake", "drs_rate"
    ]
    telemetry_sample = telemetry_sample[[column for column in telemetry_cols if column in telemetry_sample.columns]]
    master = master.merge(telemetry_sample, on=["session_key", "driver_number"], how="left")

master.to_csv(OUTPUT_TABLES / "silver_driver_session_feature_matrix.csv", index=False)
print(f"Feature matrix rows: {len(master):,}")
print(f"Feature matrix columns: {master.shape[1]:,}")
display(master.head(12))

Feature matrix rows: 1,374
Feature matrix columns: 55


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,...,overtaken_count,positions_gained,points_per_lap,avg_speed,max_speed,std_speed,avg_rpm,avg_throttle,avg_brake,drs_rate
0,1.0,1,57.0,26.0,False,False,False,5504.742,0,1229,...,9.0,0.0,0.456140,202.006905,323.535000,67.192790,9900.113307,65.787535,0.192000,0.027500
1,2.0,11,57.0,18.0,False,False,False,5527.199,22.457,1229,...,22.0,3.0,0.315789,199.343845,330.608334,68.940249,9878.458147,64.875767,0.217477,0.065783
2,3.0,55,57.0,15.0,False,False,False,5529.852,25.11,1229,...,21.0,1.0,0.263158,197.264336,327.911111,69.956170,10003.346459,66.844705,0.218099,0.063549
3,4.0,16,57.0,12.0,False,False,False,5544.411,39.669,1229,...,28.0,-2.0,0.210526,196.746998,323.000000,70.750072,10066.095324,66.603558,0.217703,0.100000
4,5.0,63,57.0,10.0,False,False,False,5551.530,46.788,1229,...,29.0,-2.0,0.175439,200.115005,325.000000,67.631084,9963.329686,64.296683,0.203540,0.033432
5,6.0,4,57.0,8.0,False,False,False,5553.200,48.458,1229,...,17.0,1.0,0.140351,196.945266,316.941828,68.966129,10277.224920,61.734717,0.217184,0.031504
6,7.0,44,57.0,6.0,False,False,False,5555.066,50.324,1229,...,15.0,2.0,0.105263,202.629487,328.645000,69.092845,9883.034614,65.054205,0.212463,0.073602
7,8.0,81,57.0,4.0,False,False,False,5560.824,56.082,1229,...,18.0,0.0,0.070175,197.151208,325.000000,69.716064,10132.586196,63.698371,0.193170,0.036697
8,9.0,14,57.0,2.0,False,False,False,5579.629,74.887,1229,...,31.0,-3.0,0.035088,196.290901,326.000000,70.076273,9906.207154,61.602331,0.211361,0.078813
9,10.0,18,57.0,1.0,False,False,False,5597.958,93.216,1229,...,25.0,2.0,0.017544,198.524564,329.558334,68.562063,9773.660213,62.752291,0.199700,0.073390


## 2. Correlation Matrix

Correlation is used as a first-pass diagnostic. Negative correlation with `finish_pos` means the feature is associated with better finishing positions because P1 is numerically lower than P20. The target columns are excluded from feature-candidate correlation to avoid treating labels as model inputs.

In [3]:
candidate_features = [
    "grid_pos", "avg_lap", "median_lap", "best_lap", "std_lap", "laps_completed",
    "outlier_laps", "pit_out_laps", "avg_i1_speed", "avg_i2_speed", "avg_st_speed",
    "air_temperature", "track_temperature", "humidity", "pressure", "wind_speed", "rainfall",
    "stint_count", "max_tyre_age_start", "avg_tyre_age_start",
    "overtakes_made", "overtaken_count", "positions_gained",
    "avg_speed", "max_speed", "std_speed", "avg_rpm", "avg_throttle", "avg_brake", "drs_rate",
]
available_features = [column for column in candidate_features if column in master.columns]
numeric_cols = available_features + ["finish_pos", "points", "target_win", "target_podium", "target_top10", "is_dnf_like"]
numeric = master[numeric_cols].apply(pd.to_numeric, errors="coerce")
corr_matrix = numeric.corr()
corr_matrix.to_csv(OUTPUT_TABLES / "silver_feature_correlation_matrix.csv")

finish_corr = (
    corr_matrix["finish_pos"]
    .drop(labels=["finish_pos"], errors="ignore")
    .dropna()
    .sort_values()
    .reset_index()
)
finish_corr.columns = ["feature", "correlation_with_finish_pos"]
finish_corr.to_csv(OUTPUT_TABLES / "finish_position_correlations.csv", index=False)
display(finish_corr)

,feature,correlation_with_finish_pos
0,target_top10,-0.857460
1,points,-0.812693
2,target_podium,-0.636984
3,target_win,-0.391008
4,positions_gained,-0.242935
5,avg_rpm,-0.163834
6,avg_speed,-0.131860
7,max_speed,-0.131782
8,std_speed,-0.100122
9,laps_completed,-0.061930


In [4]:
heatmap_cols = [column for column in [
    "finish_pos", "grid_pos", "avg_lap", "best_lap", "std_lap", "laps_completed",
    "track_temperature", "rainfall", "stint_count", "overtakes_made", "overtaken_count",
    "avg_speed", "avg_throttle", "drs_rate"
] if column in corr_matrix.columns]
fig = px.imshow(
    corr_matrix.loc[heatmap_cols, heatmap_cols],
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Silver Feature Correlation Heatmap",
)
fig.write_html(OUTPUT_CHARTS / "feature_correlation_heatmap.html", include_plotlyjs="cdn")
fig.show()

In [5]:
plot_corr = finish_corr.copy()
plot_corr["abs_corr"] = plot_corr["correlation_with_finish_pos"].abs()
plot_corr = plot_corr.sort_values("abs_corr", ascending=False).head(20).sort_values("correlation_with_finish_pos")
fig = px.bar(
    plot_corr,
    x="correlation_with_finish_pos",
    y="feature",
    orientation="h",
    color="correlation_with_finish_pos",
    color_continuous_scale="RdBu_r",
    title="Top Correlations with Finish Position",
)
fig.write_html(OUTPUT_CHARTS / "finish_position_correlations.html", include_plotlyjs="cdn")
fig.show()

## 3. Winner, Podium, and Top-10 Feature Contrast

This contrast turns correlations into interpretable race patterns. It compares robust medians for winners, podium finishers, top-10 finishers, and the rest of the field. This is more stable than raw row-level printing and gives Gold a short list of useful feature families.

In [6]:
contrast_features = [
    "grid_pos", "avg_lap", "best_lap", "std_lap", "laps_completed",
    "track_temperature", "rainfall", "stint_count", "overtakes_made",
    "avg_speed", "avg_throttle", "drs_rate"
]
contrast_features = [column for column in contrast_features if column in master.columns]
contrast_rows = []
for feature in contrast_features:
    values = pd.to_numeric(master[feature], errors="coerce")
    contrast_rows.append({
        "feature": feature,
        "winner_median": values[master["target_win"] == 1].median(),
        "non_winner_median": values[master["target_win"] == 0].median(),
        "podium_median": values[master["target_podium"] == 1].median(),
        "non_podium_median": values[master["target_podium"] == 0].median(),
        "top10_median": values[master["target_top10"] == 1].median(),
        "outside_top10_median": values[master["target_top10"] == 0].median(),
    })
contrast_df = pd.DataFrame(contrast_rows)
contrast_df["winner_delta"] = contrast_df["winner_median"] - contrast_df["non_winner_median"]
contrast_df["podium_delta"] = contrast_df["podium_median"] - contrast_df["non_podium_median"]
contrast_df["top10_delta"] = contrast_df["top10_median"] - contrast_df["outside_top10_median"]
contrast_df.to_csv(OUTPUT_TABLES / "winner_podium_top10_feature_contrast.csv", index=False)
display(contrast_df.round(3))

,feature,winner_median,non_winner_median,podium_median,non_podium_median,top10_median,outside_top10_median,winner_delta,podium_delta,top10_delta
0,grid_pos,1.000,11.000,2.000,12.000,6.000,15.000,-10.000,-10.000,-9.000
1,avg_lap,92.950,94.988,92.982,95.206,93.713,96.229,-2.038,-2.223,-2.516
2,best_lap,88.532,89.594,88.485,89.733,88.850,90.640,-1.063,-1.248,-1.790
3,std_lap,7.109,7.271,7.268,7.264,7.359,7.199,-0.162,0.004,0.160
4,laps_completed,56.000,53.000,56.000,53.000,56.000,52.000,3.000,3.000,4.000
5,track_temperature,35.408,35.214,35.408,35.214,35.408,35.214,0.194,0.194,0.194
6,rainfall,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,stint_count,2.000,2.000,2.000,2.000,2.000,2.000,0.000,0.000,0.000
8,overtakes_made,3.000,9.000,5.000,10.000,8.000,10.000,-6.000,-5.000,-2.000
9,avg_speed,186.731,185.134,186.529,185.134,176.848,193.419,1.597,1.395,-16.571


In [7]:
contrast_plot = contrast_df[["feature", "winner_delta", "podium_delta", "top10_delta"]].melt(
    id_vars="feature",
    var_name="comparison",
    value_name="median_delta",
)
fig = px.bar(
    contrast_plot,
    x="median_delta",
    y="feature",
    color="comparison",
    barmode="group",
    orientation="h",
    title="Median Feature Delta by Result Class",
)
fig.write_html(OUTPUT_CHARTS / "result_class_feature_contrast.html", include_plotlyjs="cdn")
fig.show()

## 4. Multicollinearity and Leakage Review

Silver should not silently promote target leakage into Gold. This section flags high feature-feature correlations and columns that are outcomes rather than pre-race or in-race explanatory signals.

In [8]:
feature_corr = numeric[available_features].corr()
high_corr_pairs = []
for i, left in enumerate(feature_corr.columns):
    for right in feature_corr.columns[i + 1:]:
        value = feature_corr.loc[left, right]
        if pd.notna(value) and abs(value) >= 0.70:
            high_corr_pairs.append({"feature_1": left, "feature_2": right, "correlation": value})
high_corr_df = pd.DataFrame(high_corr_pairs).sort_values("correlation", key=lambda s: s.abs(), ascending=False) if high_corr_pairs else pd.DataFrame(columns=["feature_1", "feature_2", "correlation"])
high_corr_df.to_csv(OUTPUT_TABLES / "high_correlation_pairs.csv", index=False)
display(high_corr_df.head(30))

,feature_1,feature_2,correlation
14,max_speed,avg_rpm,0.979567
15,max_speed,avg_brake,-0.978769
19,avg_rpm,avg_brake,-0.975883
2,median_lap,best_lap,0.968260
9,avg_speed,avg_rpm,0.958499
3,pit_out_laps,stint_count,0.958163
0,avg_lap,median_lap,0.958035
13,max_speed,std_speed,0.952057
10,avg_speed,avg_throttle,0.946907
17,std_speed,avg_brake,-0.931228


In [9]:
known_leakage_columns = {
    "finish_pos": "target outcome",
    "points": "post-race scoring outcome",
    "positions_gained": "uses finish_pos and grid_pos",
    "points_per_lap": "uses post-race points",
    "target_win": "label",
    "target_podium": "label",
    "target_top10": "label",
}
leakage_review = pd.DataFrame([
    {
        "column": column,
        "reason": reason,
        "gold_policy": "exclude_from_model_features",
        "allowed_for_eda": True,
    }
    for column, reason in known_leakage_columns.items()
    if column in master.columns
])
leakage_review.to_csv(OUTPUT_TABLES / "leakage_review.csv", index=False)
display(leakage_review)

,column,reason,gold_policy,allowed_for_eda
0,finish_pos,target outcome,exclude_from_model_features,True
1,points,post-race scoring outcome,exclude_from_model_features,True
2,positions_gained,uses finish_pos and grid_pos,exclude_from_model_features,True
3,points_per_lap,uses post-race points,exclude_from_model_features,True
4,target_win,label,exclude_from_model_features,True
5,target_podium,label,exclude_from_model_features,True
6,target_top10,label,exclude_from_model_features,True


## 5. Feature Selection Recommendations

The final recommendation table separates safe candidate features from EDA-only outcome columns. Gold feature engineering should prefer features that are available before or during the prediction timestamp and should avoid direct outcome-derived columns.

In [10]:
finish_corr_map = finish_corr.set_index("feature")["correlation_with_finish_pos"].to_dict()
safe_feature_candidates = [
    feature for feature in available_features
    if feature not in {"positions_gained", "points_per_lap"}
]
recommendations = []
for feature in safe_feature_candidates:
    corr_value = finish_corr_map.get(feature, np.nan)
    abs_corr = abs(corr_value) if pd.notna(corr_value) else np.nan
    if feature in {"grid_pos", "avg_lap", "median_lap", "best_lap", "std_lap", "laps_completed"}:
        family = "race_pace"
    elif feature in {"air_temperature", "track_temperature", "humidity", "pressure", "wind_speed", "rainfall"}:
        family = "weather_context"
    elif feature in {"stint_count", "max_tyre_age_start", "avg_tyre_age_start"}:
        family = "strategy_tyre"
    elif feature in {"avg_speed", "max_speed", "std_speed", "avg_rpm", "avg_throttle", "avg_brake", "drs_rate"}:
        family = "sampled_telemetry"
    else:
        family = "racecraft"
    recommendations.append({
        "feature": feature,
        "family": family,
        "correlation_with_finish_pos": corr_value,
        "abs_correlation": abs_corr,
        "recommendation": "KEEP" if pd.notna(abs_corr) and abs_corr >= 0.10 else "REVIEW",
    })
feature_recommendations = pd.DataFrame(recommendations).sort_values("abs_correlation", ascending=False)
feature_recommendations.to_csv(OUTPUT_TABLES / "gold_feature_recommendations.csv", index=False)
display(feature_recommendations.head(30))

,feature,family,correlation_with_finish_pos,abs_correlation,recommendation
0,grid_pos,race_pace,0.741567,0.741567,KEEP
21,overtaken_count,racecraft,0.289043,0.289043,KEEP
20,overtakes_made,racecraft,0.188887,0.188887,KEEP
25,avg_rpm,sampled_telemetry,-0.163834,0.163834,KEEP
27,avg_brake,sampled_telemetry,0.157884,0.157884,KEEP
22,avg_speed,sampled_telemetry,-0.131860,0.131860,KEEP
23,max_speed,sampled_telemetry,-0.131782,0.131782,KEEP
24,std_speed,sampled_telemetry,-0.100122,0.100122,KEEP
2,median_lap,race_pace,0.063248,0.063248,REVIEW
3,best_lap,race_pace,0.062128,0.062128,REVIEW


In [11]:
fig = px.bar(
    feature_recommendations.head(25).sort_values("abs_correlation"),
    x="abs_correlation",
    y="feature",
    color="family",
    orientation="h",
    title="Gold Candidate Feature Strength by Family",
)
fig.write_html(OUTPUT_CHARTS / "gold_feature_recommendations.html", include_plotlyjs="cdn")
fig.show()

## Final Silver EDA Gate

The Silver EDA layer is complete when all seven notebooks have produced reports, a compact feature matrix exists, and leakage-sensitive columns are documented before Gold engineering begins.

In [12]:
safe_finish_corr = finish_corr[finish_corr["feature"].isin(safe_feature_candidates)].copy()
top_finish_feature = safe_finish_corr.iloc[safe_finish_corr["correlation_with_finish_pos"].abs().argmax()] if len(safe_finish_corr) else None
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "status": "PASS",
    "feature_matrix_rows": int(len(master)),
    "feature_matrix_columns": int(master.shape[1]),
    "candidate_feature_count": int(len(safe_feature_candidates)),
    "high_correlation_pair_count": int(len(high_corr_df)),
    "leakage_review_columns": int(len(leakage_review)),
    "top_safe_finish_corr_feature": top_finish_feature["feature"] if top_finish_feature is not None else None,
    "top_safe_finish_corr_value": float(top_finish_feature["correlation_with_finish_pos"]) if top_finish_feature is not None else 0.0,
}
write_report("correlation_analysis", report)
write_insight(
    "Silver Correlation Analysis Insights",
    [
        f"Built a compact driver-session matrix with {report['feature_matrix_rows']:,} rows and {report['feature_matrix_columns']} columns.",
        f"Reviewed {report['candidate_feature_count']} candidate model features.",
        f"Strongest safe finish-position correlation: {report['top_safe_finish_corr_feature']} = {report['top_safe_finish_corr_value']:.3f}.",
    ],
    [
        f"{len(high_corr_df)} high-correlation feature pairs need Gold feature selection review.",
        "Outcome-derived columns such as points and positions_gained are documented as EDA-only leakage risks.",
    ],
    [
        "Use grid, lap pace, stint, weather, racecraft, and sampled telemetry families as Gold inputs.",
        "Exclude labels and post-race outcome columns from training features.",
        "Resolve high-correlation pairs before model training to reduce redundant signal.",
    ],
)
(CHECKPOINTS / "silver_eda_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
print("Silver EDA completed. Ready for Gold feature engineering.")

{'notebook': '07_correlation_analysis', 'timestamp': '2026-06-02T00:46:47.226842', 'status': 'PASS', 'feature_matrix_rows': 1374, 'feature_matrix_columns': 55, 'candidate_feature_count': 29, 'high_correlation_pair_count': 21, 'leakage_review_columns': 7, 'top_safe_finish_corr_feature': 'grid_pos', 'top_safe_finish_corr_value': 0.7415668342234911}
Silver EDA completed. Ready for Gold feature engineering.
